In [1]:
import pickle
import bz2
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from ipywidgets import interact, IntSlider

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
example_file = "C:/Users/docto/Desktop/myosegmenTUM/FATFRACTION_collected/HV005_1_FATFRACTION/HV005_1_FATFRACTION_stack1.nii.rois.p"

In [ ]:
import io

class _GenericObject:
    """Stand-in for any dafne class not available in this environment."""
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)
        if args:
            self._args = args
    def __repr__(self):
        return f"{self.__class__.__name__}({self.__dict__})"

class _SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if "dafne" in module:
            # create a named subclass so the original class name is preserved
            return type(name, (_GenericObject,), {})
        return super().find_class(module, name)

with bz2.open(example_file, "rb") as f:
    raw = f.read()

rois = _SafeUnpickler(io.BytesIO(raw)).load()

print(type(rois))
if isinstance(rois, dict):
    print("Keys:", list(rois.keys()))
elif isinstance(rois, (list, tuple)):
    print(f"Length: {len(rois)}, first element type: {type(rois[0])}")
else:
    print(vars(rois) if hasattr(rois, "__dict__") else rois)

In [ ]:
roi_manager = rois['roiManager']
all_rois = roi_manager.allROIs
mask_size = roi_manager.mask_size  # (height, width)

print("Muscles found:", list(all_rois.keys()))
print("Mask size (H, W):", mask_size)

# Determine number of slices
all_slice_nums = {s for slices in all_rois.values() for s in slices.keys()}
num_slices = max(all_slice_nums) + 1
print(f"Slices: {sorted(all_slice_nums)}")

# Build 3D segmentation array: shape = (slices, H, W)
# Each muscle gets a unique integer label; overlaps keep the last-written label.
segmentation = np.zeros((num_slices, mask_size[0], mask_size[1]), dtype=np.uint16)
label_map = {}

for label_idx, (muscle_name, slices_dict) in enumerate(all_rois.items(), start=1):
    label_map[label_idx] = muscle_name
    for slice_num, pair in slices_dict.items():
        if pair.mask is None:
            continue
        # pair.mask is bz2+pickle compressed — decompress both layers
        mask = pickle.loads(bz2.decompress(pair.mask))
        if mask is None:
            continue
        mask = np.asarray(mask, dtype=np.uint8)
        segmentation[slice_num][mask > 0] = label_idx

print("\nSegmentation shape:", segmentation.shape)
print("Label map:", label_map)
print("Unique labels in segmentation:", np.unique(segmentation))

In [ ]:
!pwd

In [ ]:
ground_truth_segmentation = "../myosegmenTUM/HV005_1/SegmentationMasks/combined_gt_stack1.mha"
ground_truth = "../myosegmenTUM/HV005_1/ImageData/HV005_1_FATFRACTION/HV005_1_FATFRACTION_stack1.nii"

In [ ]:


# --- load the ground truth image ---
gt_sitk = sitk.ReadImage(ground_truth)
gt_array = sitk.GetArrayFromImage(gt_sitk).astype(float)  # shape: (slices, H, W)

# normalise to [0, 1] for display
gt_norm = (gt_array - gt_array.min()) / (gt_array.max() - gt_array.min() + 1e-8)

print("GT image shape :", gt_array.shape)
print("Segmentation shape:", segmentation.shape)

# --- build a per-voxel RGBA overlay from the segmentation ---
cmap = plt.cm.get_cmap("tab10", len(label_map))
overlay = np.zeros((*segmentation.shape, 4), dtype=float)  # RGBA
for i, (label_idx, name) in enumerate(label_map.items()):
    color = cmap(i)
    overlay[segmentation == label_idx] = [color[0], color[1], color[2], 0.5]

# --- legend handles ---
legend_patches = [
    mpatches.Patch(color=cmap(i), alpha=0.5, label=name)
    for i, (_, name) in enumerate(label_map.items())
]

# --- scrollable viewer ---
n_slices = min(gt_norm.shape[0], segmentation.shape[0])

def show_slice(slice_idx):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    img = gt_norm[slice_idx]
    seg_overlay = overlay[slice_idx]

    axes[0].imshow(img, cmap="gray", origin="lower")
    axes[0].set_title(f"Ground truth image  —  slice {slice_idx}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray", origin="lower")
    axes[1].imshow(seg_overlay, origin="lower")
    axes[1].set_title(f"+ Dafne segmentation  —  slice {slice_idx}")
    axes[1].axis("off")
    axes[1].legend(handles=legend_patches, loc="lower right", fontsize=7,
                   framealpha=0.7)

    plt.tight_layout()
    plt.show()

interact(show_slice, slice_idx=IntSlider(min=0, max=n_slices - 1, step=1, value=0,
                                         description="Slice"))